# Exercise for Unit 4.1 Naïve Bayes

**Name:** RALPH MARTIN CHUA  
**Date:** February 27, 2026

## Dataset

Training dataset with documents and their class labels:

| Document | Class |
|----------|-------|
| Free money now!!! | SPAM |
| Hi mom, how are you? | HAM |
| Lowest price for your meds | SPAM |
| Are we still on for dinner? | HAM |
| Win a free iPhone today | SPAM |
| Let's catch up tomorrow at the office | HAM |
| Meeting at 3 PM tomorrow | HAM |
| Get 50% off, limited time! | SPAM |
| Team meeting in the office | HAM |
| Click here for prizes! | SPAM |
| Can you send the report? | HAM |

In [1]:
# Dataset
documents = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

# Test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

print(f"Training dataset: {len(documents)} documents")
print(f"Test sentences: {len(test_sentences)} sentences")

Training dataset: 11 documents
Test sentences: 2 sentences


---

## 1: Manual Implementation of Naïve Bayes

### Task 1a: Generate Bag of Words

In [2]:
import re
from collections import defaultdict, Counter
import math

def preprocess_text(text):
    """Convert text to lowercase and extract words (alphanumeric only)"""
    text = text.lower()
    words = re.findall(r'\b[a-z0-9]+\b', text)
    return words

def generate_bag_of_words(documents):
    """
    Generate a Bag of Words for word frequency.
    
    Returns:
    - vocabulary: set of all unique words
    - word_freq_by_class: word frequencies per class
    - class_word_counts: total word count per class
    """
    vocabulary = set()
    word_freq_by_class = defaultdict(lambda: defaultdict(int))
    class_word_counts = defaultdict(int)
    
    for doc, label in documents:
        words = preprocess_text(doc)
        vocabulary.update(words)
        
        for word in words:
            word_freq_by_class[label][word] += 1
            class_word_counts[label] += 1
    
    return vocabulary, word_freq_by_class, class_word_counts

vocabulary, word_freq_by_class, class_word_counts = generate_bag_of_words(documents)

print("Vocabulary:", sorted(vocabulary))
print(f"\nVocabulary size: {len(vocabulary)}")
print(f"\nWord frequencies by class:")
for label in ['HAM', 'SPAM']:
    print(f"\n{label}:")
    print(dict(word_freq_by_class[label]))
print(f"\nTotal word count per class:")
print(f"HAM: {class_word_counts['HAM']}")
print(f"SPAM: {class_word_counts['SPAM']}")

Vocabulary: ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

Vocabulary size: 45

Word frequencies by class:

HAM:
{'hi': 1, 'mom': 1, 'how': 1, 'are': 2, 'you': 2, 'we': 1, 'still': 1, 'on': 1, 'for': 1, 'dinner': 1, 'let': 1, 's': 1, 'catch': 1, 'up': 1, 'tomorrow': 2, 'at': 2, 'the': 3, 'office': 2, 'meeting': 2, '3': 1, 'pm': 1, 'team': 1, 'in': 1, 'can': 1, 'send': 1, 'report': 1}

SPAM:
{'free': 2, 'money': 1, 'now': 1, 'lowest': 1, 'price': 1, 'for': 2, 'your': 1, 'meds': 1, 'win': 1, 'a': 1, 'iphone': 1, 'today': 1, 'get': 1, '50': 1, 'off': 1, 'limited': 1, 'time': 1, 'click': 1, 'here': 1, 'prizes': 1}

Total word count per class:
HAM: 34
SPAM: 22


### Task 1b: Calculate Prior Probabilities


In [3]:
def calculate_prior(documents):
    """
    Calculate prior probabilities for each class.
    P(class) = count of documents in class / total documents
    """
    class_counts = Counter(label for _, label in documents)
    total_docs = len(documents)
    
    priors = {label: count / total_docs for label, count in class_counts.items()}
    return priors, class_counts

priors, class_counts = calculate_prior(documents)

print("Class counts:")
for label in ['HAM', 'SPAM']:
    print(f"{label}: {class_counts[label]}")
    
print("\nPrior probabilities:")
for label in ['HAM', 'SPAM']:
    print(f"P({label}) = {class_counts[label]}/{len(documents)} = {priors[label]:.4f}")

Class counts:
HAM: 6
SPAM: 5

Prior probabilities:
P(HAM) = 6/11 = 0.5455
P(SPAM) = 5/11 = 0.4545


### Task 1c: Calculate Likelihood of Tokens


In [4]:
def calculate_likelihood(word, label, word_freq_by_class, class_word_counts, vocabulary):
    """
    Calculate likelihood of a word given a class using Laplace smoothing.
    P(word|class) = (count of word in class + 1) / (total words in class + |V|)
    """
    vocab_size = len(vocabulary)
    word_count_in_class = word_freq_by_class[label].get(word, 0)
    total_words_in_class = class_word_counts[label]
    
    likelihood = (word_count_in_class + 1) / (total_words_in_class + vocab_size)
    return likelihood

# Display likelihood for some sample words
print("Sample likelihoods (with Laplace smoothing):\n")
sample_words = ['free', 'meeting', 'offer', 'office', 'money']

for word in sample_words:
    if word in vocabulary:
        print(f"Word: '{word}'")
        for label in ['HAM', 'SPAM']:
            lik = calculate_likelihood(word, label, word_freq_by_class, class_word_counts, vocabulary)
            count = word_freq_by_class[label].get(word, 0)
            print(f"  P({word}|{label}) = ({count} + 1) / ({class_word_counts[label]} + {len(vocabulary)}) = {lik:.6f}")
        print()

Sample likelihoods (with Laplace smoothing):

Word: 'free'
  P(free|HAM) = (0 + 1) / (34 + 45) = 0.012658
  P(free|SPAM) = (2 + 1) / (22 + 45) = 0.044776

Word: 'meeting'
  P(meeting|HAM) = (2 + 1) / (34 + 45) = 0.037975
  P(meeting|SPAM) = (0 + 1) / (22 + 45) = 0.014925

Word: 'office'
  P(office|HAM) = (2 + 1) / (34 + 45) = 0.037975
  P(office|SPAM) = (0 + 1) / (22 + 45) = 0.014925

Word: 'money'
  P(money|HAM) = (0 + 1) / (34 + 45) = 0.012658
  P(money|SPAM) = (1 + 1) / (22 + 45) = 0.029851



### Task 1d: Classify Test Sentences



In [5]:
def classify_naive_bayes(sentence, priors, word_freq_by_class, class_word_counts, vocabulary):
    """
    Classify a sentence using Naïve Bayes.
    Returns the predicted class and log probabilities for each class.
    """
    words = preprocess_text(sentence)
    log_probs = {}
    
    for label in priors:
        # Start with log of prior
        log_prob = math.log(priors[label])
        
        # Add log likelihood for each word
        for word in words:
            likelihood = calculate_likelihood(word, label, word_freq_by_class, class_word_counts, vocabulary)
            log_prob += math.log(likelihood)
        
        log_probs[label] = log_prob
    
    # Predict class with highest log probability
    predicted_class = max(log_probs, key=log_probs.get)
    
    return predicted_class, log_probs

# Classify test sentences
print("Manual Naïve Bayes Classification Results:\n")
print("=" * 60)

for i, sentence in enumerate(test_sentences, 1):
    print(f"\nTest Sentence {i}: \"{sentence}\"")
    print("-" * 60)
    
    predicted_class, log_probs = classify_naive_bayes(
        sentence, priors, word_freq_by_class, class_word_counts, vocabulary
    )
    
    print(f"\nWords in sentence: {preprocess_text(sentence)}")
    print(f"\nLog Probabilities:")
    for label in ['HAM', 'SPAM']:
        print(f"  log P({label}|sentence) = {log_probs[label]:.6f}")
    
    print(f"\n✓ Predicted Class: {predicted_class}")
    print()

Manual Naïve Bayes Classification Results:


Test Sentence 1: "Limited offer, click here!"
------------------------------------------------------------

Words in sentence: ['limited', 'offer', 'click', 'here']

Log Probabilities:
  log P(HAM|sentence) = -18.083927
  log P(SPAM|sentence) = -15.527786

✓ Predicted Class: SPAM


Test Sentence 2: "Meeting at 2 PM with the manager."
------------------------------------------------------------

Words in sentence: ['meeting', 'at', '2', 'pm', 'with', 'the', 'manager']

Log Probabilities:
  log P(HAM|sentence) = -26.915605
  log P(SPAM|sentence) = -30.221306

✓ Predicted Class: HAM



---

## 2: Using Scikit-Learn

### Multinomial Naïve Bayes Classifier with CountVectorizer

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import numpy as np

# Prepare training data
X_train = [doc for doc, _ in documents]
y_train = [label for _, label in documents]

print("Training Data:")
print(f"Number of documents: {len(X_train)}")
print(f"Class distribution: {Counter(y_train)}")

Training Data:
Number of documents: 11
Class distribution: Counter({'HAM': 6, 'SPAM': 5})


In [7]:
# Create and train the model
pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB())
])

# Train the model
pipeline.fit(X_train, y_train)

print("Model trained successfully!")
print(f"\nVocabulary size: {len(pipeline.named_steps['vectorizer'].vocabulary_)}")
print(f"Feature names (all): {sorted(pipeline.named_steps['vectorizer'].get_feature_names_out())}")

Model trained successfully!

Vocabulary size: 42
Feature names (all): ['50', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']


### Task 2a: Classify Test Sentences with Scikit-Learn

In [8]:
# Classify test sentences
print("Scikit-Learn Naïve Bayes Classification Results:\n")
print("=" * 60)

for i, sentence in enumerate(test_sentences, 1):
    print(f"\nTest Sentence {i}: \"{sentence}\"")
    print("-" * 60)
    
    # Predict
    prediction = pipeline.predict([sentence])[0]
    
    # Get probability scores
    proba = pipeline.predict_proba([sentence])[0]
    classes = pipeline.classes_
    
    print(f"\nProbabilities:")
    for cls, prob in zip(classes, proba):
        print(f"  P({cls}|sentence) = {prob:.6f}")
    
    print(f"\n✓ Predicted Class: {prediction}")
    print()

Scikit-Learn Naïve Bayes Classification Results:


Test Sentence 1: "Limited offer, click here!"
------------------------------------------------------------

Probabilities:
  P(HAM|sentence) = 0.084717
  P(SPAM|sentence) = 0.915283

✓ Predicted Class: SPAM


Test Sentence 2: "Meeting at 2 PM with the manager."
------------------------------------------------------------

Probabilities:
  P(HAM|sentence) = 0.978443
  P(SPAM|sentence) = 0.021557

✓ Predicted Class: HAM



---

## Comparison: Manual vs Scikit-Learn Results

In [9]:
# Compare results side by side
print("Comparison of Manual vs Scikit-Learn Classifications:\n")
print("=" * 70)

for i, sentence in enumerate(test_sentences, 1):
    # Manual prediction
    manual_pred, _ = classify_naive_bayes(
        sentence, priors, word_freq_by_class, class_word_counts, vocabulary
    )
    
    # Scikit-Learn prediction
    sklearn_pred = pipeline.predict([sentence])[0]
    
    match = "✓" if manual_pred == sklearn_pred else "✗"
    
    print(f"\nTest {i}: \"{sentence}\"")
    print(f"  Manual Implementation: {manual_pred}")
    print(f"  Scikit-Learn:          {sklearn_pred}")
    print(f"  Match: {match}")

Comparison of Manual vs Scikit-Learn Classifications:


Test 1: "Limited offer, click here!"
  Manual Implementation: SPAM
  Scikit-Learn:          SPAM
  Match: ✓

Test 2: "Meeting at 2 PM with the manager."
  Manual Implementation: HAM
  Scikit-Learn:          HAM
  Match: ✓
